# HIPAA Privacy Rule — Demo Notebook
---

## 1. Normal Data

This is how data looks in any basic system — a simple table with user information.
No access control. No restrictions. Anyone can see everything.


In [2]:
import pandas as pd
from IPython.display import display

normal_data = pd.DataFrame([
    {"ID": "001", "Name": "Ravi Sharma",  "Email": "ravi@email.com",  "Phone": "9876543210", "Age": 34, "City": "Pune"},
    {"ID": "002", "Name": "Priya Mehta",  "Email": "priya@email.com", "Phone": "9823456789", "Age": 29, "City": "Mumbai"},
    {"ID": "003", "Name": "Anil Desai",   "Email": "anil@email.com",  "Phone": "9812345678", "Age": 45, "City": "Delhi"},
])

print("Normal User Data — No Rules Applied")
print("Who can see this? EVERYONE.\n")
display(normal_data)

Normal User Data — No Rules Applied
Who can see this? EVERYONE.



,ID,Name,Email,Phone,Age,City
0,001,Ravi Sharma,ravi@email.com,9876543210,34,Pune
1,002,Priya Mehta,priya@email.com,9823456789,29,Mumbai
2,003,Anil Desai,anil@email.com,9812345678,45,Delhi


---
## 2. The Problem

When **medical data** is handled the same way — no restrictions, no accountability — real harm happens.

| Case | What Happened | Consequence |
|------|--------------|-------------|
| Employer Discrimination | Receptionist shared patient's HIV diagnosis with employer | Patient got fired |
| Data Sold | Clinic sold patient records to pharma company | No patient consent, used for targeted ads |
| Unauthorized Access | Nurse accessed celebrity patient's records out of curiosity | No log, no consequence, privacy violated |
| Wrong Recipient | Hospital faxed test results to wrong number | Mental health diagnosis exposed to stranger |

Without rules, medical data looks like this:

In [3]:
unprotected_medical = pd.DataFrame([
    {"Patient": "Ravi Sharma",  "Diagnosis": "HIV Positive",   "Medication": "Antiretroviral", "Insurance": "Star Health",  "Who Can See?": "EVERYONE"},
    {"Patient": "Priya Mehta",  "Diagnosis": "Depression",      "Medication": "Sertraline",     "Insurance": "HDFC Ergo",    "Who Can See?": "EVERYONE"},
    {"Patient": "Anil Desai",   "Diagnosis": "Diabetes Type 2", "Medication": "Metformin",      "Insurance": "Max Bupa",     "Who Can See?": "EVERYONE"},
])

print("Medical Data — No Rules Applied\n")
display(unprotected_medical)

Medical Data — No Rules Applied



,Patient,Diagnosis,Medication,Insurance,Who Can See?
0,Ravi Sharma,HIV Positive,Antiretroviral,Star Health,EVERYONE
1,Priya Mehta,Depression,Sertraline,HDFC Ergo,EVERYONE
2,Anil Desai,Diabetes Type 2,Metformin,Max Bupa,EVERYONE


---
## 3. What is PHI — Protected Health Information

**PHI** is any information that can **identify a patient** AND relates to their **health condition, treatment, or payment**.

> **Formula:** Identity Information + Health/Treatment/Payment Information = PHI

### The 18 HIPAA Identifiers (key ones)
- Full Name
- Date of Birth
- Phone Number
- Email Address
- Social Security Number
- Medical Record Number
- IP Address
- Geographic Data
- Photographs / Biometrics


In [4]:
phi_examples = pd.DataFrame([
    {"Example": "Ravi Sharma has Diabetes",             "Is PHI?": "YES — identifies person + health condition"},
    {"Example": "5% of patients have Diabetes",         "Is PHI?": "NO  — no individual identified"},
    {"Example": "Patient #001 prescribed Metformin",    "Is PHI?": "YES — record number links to individual"},
    {"Example": "Metformin is used for Diabetes",       "Is PHI?": "NO  — general medical knowledge"},
    {"Example": "IP 192.168.1.1 accessed cancer record","Is PHI?": "YES — IP address is an identifier"},
])

print("PHI vs Not PHI\n")
display(phi_examples)

PHI vs Not PHI



,Example,Is PHI?
0,Ravi Sharma has Diabetes,YES — identifies person + health condition
1,5% of patients have Diabetes,NO — no individual identified
2,Patient #001 prescribed Metformin,YES — record number links to individual
3,Metformin is used for Diabetes,NO — general medical knowledge
4,IP 192.168.1.1 accessed cancer record,YES — IP address is an identifier


---
## 4. HIPAA — Privacy Rule

**HIPAA** (Health Insurance Portability and Accountability Act, 1996) sets the standard for protecting patient data.

The **Privacy Rule** specifically governs how PHI can be used and disclosed.

### Key Requirements

| Requirement | What It Means |
|-------------|---------------|
| Minimum Necessary Standard | Only access the minimum PHI needed for the task |
| Patient Authorization | Cannot share PHI with third parties without patient consent |
| Patient Access Rights | Patients can view and get copies of their own records |
| Amendment Rights | Patients can request corrections to inaccurate records |
| Disclosure Accounting | Log of all PHI disclosures must be maintained |
| Notice of Privacy Practices | Patients must be informed how their data is used |

**Exceptions where sharing is allowed without consent (TPO):**
- **T**reatment — sharing with other treating physicians
- **P**ayment — billing insurance
- **O**perations — internal hospital quality management

---
## 5. Demo — Scenario Based

**Scenario:** City General Hospital. Patient Ravi Sharma visits for a checkup.

<!-- We will walk through 6 scenarios, each demonstrating a specific Privacy Rule requirement. -->

In [5]:
# ── Setup: Patient Database ──────────────────────────────────────────

patients = {
    "P001": {
        "name":        "Ravi Sharma",
        "dob":         "15-Mar-1990",
        "phone":       "98765-43210",
        "diagnosis":   "Type 2 Diabetes",
        "medication":  "Metformin 500mg",
        "insurance":   "Star Health #SH4521",
        "last_visit":  "12-Apr-2026",
        "doctor":      "Dr. Amit"
    },
    "P002": {
        "name":        "Priya Mehta",
        "dob":         "08-Jul-1995",
        "phone":       "98234-56789",
        "diagnosis":   "Depression",
        "medication":  "Sertraline 50mg",
        "insurance":   "HDFC Ergo #HD7823",
        "last_visit":  "10-Apr-2026",
        "doctor":      "Dr. Amit"
    }
}

# ── Role Permissions: which fields each role can access ──────────────

role_permissions = {
    "Doctor":        ["name", "dob", "phone", "diagnosis", "medication", "insurance", "last_visit"],
    "Nurse":         ["name", "dob", "diagnosis", "medication", "last_visit"],
    "Receptionist":  ["name", "phone", "last_visit"],
    "Patient":       ["name", "dob", "phone", "diagnosis", "medication", "insurance", "last_visit"],
    "Admin":         ["name", "dob", "phone", "diagnosis", "medication", "insurance", "last_visit"],
}

# ── Disclosure / Audit Log ───────────────────────────────────────────

disclosure_log = []

def log(user, role, action, status):
    from datetime import datetime
    entry = {
        "Time":   datetime.now().strftime("%H:%M:%S"),
        "User":   user,
        "Role":   role,
        "Action": action,
        "Status": status
    }
    disclosure_log.append(entry)

# ── Access Function ──────────────────────────────────────────────────

def access_record(user, role, patient_id):
    allowed_fields = role_permissions.get(role, [])
    patient = patients.get(patient_id)

    if not patient:
        print("Patient not found.")
        return

    print(f"User    : {user} ({role})")
    print(f"Patient : {patient['name']} [{patient_id}]")
    print("-" * 45)

    all_fields = ["name", "dob", "phone", "diagnosis", "medication", "insurance", "last_visit"]
    for field in all_fields:
        if field in allowed_fields:
            print(f"  {field.upper():<15}: {patient[field]}")
        else:
            print(f"  {field.upper():<15}: [RESTRICTED — Minimum Necessary Standard]")

    log(user, role, f"Accessed record of {patient['name']}", "ALLOWED")
    print()

print("Setup complete. Patient database and roles loaded.")
print(f"Patients loaded  : {list(patients.keys())}")
print(f"Roles configured : {list(role_permissions.keys())}")

Setup complete. Patient database and roles loaded.
Patients loaded  : ['P001', 'P002']
Roles configured : ['Doctor', 'Nurse', 'Receptionist', 'Patient', 'Admin']


---
### Scenario 1 — Doctor Views Patient Record
**Privacy Rule:** Authorized access for treatment (TPO). Doctor sees all fields.

In [6]:
print("=" * 50)
print("SCENARIO 1 — Doctor Accesses Patient Record")
print("Privacy Rule: Treatment falls under TPO — full access permitted")
print("=" * 50)
print()

access_record(user="Dr. Amit", role="Doctor", patient_id="P001")

SCENARIO 1 — Doctor Accesses Patient Record
Privacy Rule: Treatment falls under TPO — full access permitted

User    : Dr. Amit (Doctor)
Patient : Ravi Sharma [P001]
---------------------------------------------
  NAME           : Ravi Sharma
  DOB            : 15-Mar-1990
  PHONE          : 98765-43210
  DIAGNOSIS      : Type 2 Diabetes
  MEDICATION     : Metformin 500mg
  INSURANCE      : Star Health #SH4521
  LAST_VISIT     : 12-Apr-2026



---
### Scenario 2 — Receptionist Tries to Access Full Record
**Privacy Rule:** Minimum Necessary Standard — receptionist only needs name, phone, and last visit to schedule appointments. Diagnosis and medication are restricted.

In [7]:
print("=" * 50)
print("SCENARIO 2 — Receptionist Accesses Patient Record")
print("Privacy Rule: Minimum Necessary Standard applied")
print("=" * 50)
print()

access_record(user="Raj", role="Receptionist", patient_id="P001")

print(">> Raj only needs Name, Phone, Last Visit to schedule an appointment.")
print(">> Diagnosis and Medication are NOT necessary for this task — restricted.")

SCENARIO 2 — Receptionist Accesses Patient Record
Privacy Rule: Minimum Necessary Standard applied

User    : Raj (Receptionist)
Patient : Ravi Sharma [P001]
---------------------------------------------
  NAME           : Ravi Sharma
  DOB            : [RESTRICTED — Minimum Necessary Standard]
  PHONE          : 98765-43210
  DIAGNOSIS      : [RESTRICTED — Minimum Necessary Standard]
  MEDICATION     : [RESTRICTED — Minimum Necessary Standard]
  INSURANCE      : [RESTRICTED — Minimum Necessary Standard]
  LAST_VISIT     : 12-Apr-2026

>> Raj only needs Name, Phone, Last Visit to schedule an appointment.
>> Diagnosis and Medication are NOT necessary for this task — restricted.


---
### Scenario 3 — Patient Views Own Record
**Privacy Rule:** Patients have the right to access their own complete records.

In [8]:
print("=" * 50)
print("SCENARIO 3 — Patient Accesses Own Record")
print("Privacy Rule: Patient Access Rights — full record granted")
print("=" * 50)
print()

access_record(user="Ravi Sharma", role="Patient", patient_id="P001")

print(">> Patient has the right to view, download, and request copies of their own records.")

SCENARIO 3 — Patient Accesses Own Record
Privacy Rule: Patient Access Rights — full record granted

User    : Ravi Sharma (Patient)
Patient : Ravi Sharma [P001]
---------------------------------------------
  NAME           : Ravi Sharma
  DOB            : 15-Mar-1990
  PHONE          : 98765-43210
  DIAGNOSIS      : Type 2 Diabetes
  MEDICATION     : Metformin 500mg
  INSURANCE      : Star Health #SH4521
  LAST_VISIT     : 12-Apr-2026

>> Patient has the right to view, download, and request copies of their own records.
